In [ ]:
# The straggler tax: why long-horizon rollouts waste GPUs
import numpy as np
rng = np.random.default_rng(7)

N = 200                      # executions per RL step
d = rng.lognormal(mean=0.0, sigma=0.9, size=N)   # hours per execution (heavy tail)
rollout_work = d.sum()       # useful GPU-hours of rollout work
max_d = d.max()
n_stragglers = int((d > 2.0).sum())
print(f"{N} executions, useful rollout work {rollout_work:.1f} GPU-h")
print(f"median {np.median(d):.2f}h, max {max_d:.2f}h, stragglers >2h: {n_stragglers}")

POOL = 96
train_work = 48.0            # GPU-hours of training work per step
useful = rollout_work + train_work

# Colocate: whole pool runs rollouts (waits for the slowest), then whole pool trains
wall = {}
wall['Colocate'] = max_d + train_work / POOL

# Async: fixed 48/48 split; rollout engines suffer 10% packing fragmentation
wall['Async'] = max(rollout_work * 1.10 / 48, train_work / 48)

# Elastic (QwenGyre): cells migrate to the work; ~0.5% switching overhead
wall['Elastic'] = (rollout_work * 1.005 + train_work) / POOL

idle = {k: POOL * v - useful for k, v in wall.items()}
for k in ('Colocate', 'Async', 'Elastic'):
    print(f"{k:9s}: wall {wall[k]:.2f}h, wasted GPU-h {idle[k]:.1f}")
assert idle['Elastic'] < idle['Async'] < idle['Colocate']
assert idle['Elastic'] / idle['Colocate'] < 0.6
print("OK: the elastic scheduler wastes the fewest GPU-hours")

In [ ]:
# Prefix-shared trajectory trees: dedupe the shared prefix
# one execution: 30 shared turns, then b branches of 20 unique turns each
prefix, branch_len = 30, 20
for b in (2, 4, 8):
    naive = b * (prefix + branch_len)
    shared = prefix + b * branch_len          # prefix computed once
    capped = prefix + min(b, 5) * branch_len  # QwenGyre: <=5 trajectories/execution
    print(f"branches={b}: naive {naive}, prefix-shared {shared} "
          f"({naive/shared:.2f}x), 5-capped {capped} ({naive/capped:.2f}x)")
assert (8 * 50) / (30 + 8 * 20) > 2.0        # prefix sharing beats 2x at 8 branches
assert (8 * 50) / (30 + 5 * 20) > 3.0        # the 5-trajectory cap beats 3x
print("OK: trajectory trees bound training cost")

In [ ]:
# Partial-progress scoring: timed-out runs still teach
rng = np.random.default_rng(11)
n_timeout = 50
true_skill = rng.uniform(0.3, 1.0, n_timeout)          # latent capability
progress = np.clip(true_skill + rng.normal(0, 0.1, n_timeout), 0, 1)
naive = np.zeros(n_timeout)        # timed out -> reward 0, all signal lost
graded = progress                  # score the preserved workspace instead
print(f"naive mean reward  {naive.mean():.3f}  (signal lost)")
print(f"graded mean reward {graded.mean():.3f}, corr with true skill "
      f"{np.corrcoef(graded, true_skill)[0, 1]:.3f}")
assert naive.mean() == 0.0
assert graded.mean() > 0.3
assert np.corrcoef(graded, true_skill)[0, 1] > 0.8
assert (graded > 0).sum() >= int(0.9 * n_timeout)
print("OK: scoring preserved workspaces recovers the learning signal")

In [ ]:
# The paper's headline numbers, recomputed (arXiv 2609.33848)
gyre, async_t, colo_t = 75.42, 134.55, 91.47
print(f"75.42h vs Async 134.55h -> {async_t / gyre:.2f}x")
print(f"75.42h vs Colocate 91.47h -> {colo_t / gyre:.2f}x")
nl2_start, nl2_end = 52.48, 58.54
print(f"NL2RepoBench {nl2_start:.2f}% -> {nl2_end:.2f}% = +{nl2_end - nl2_start:.2f}pp in 48 steps")
print("cell role-switch: 8.52s rollout->train, 3.46s train->rollout (negligible vs hour-long rollouts)")
assert abs(async_t / gyre - 1.78) < 0.01
assert abs(colo_t / gyre - 1.21) < 0.01
assert abs((nl2_end - nl2_start) - 6.06) < 0.01
print("OK: speedups and the 6pp gain reproduce")